In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# -----------------------------
# USER INPUTS
# -----------------------------
INPUT_FILE = Path(r"C:\Users\Martin\Documents\GitHub\LB_DEV\data\District\Theme 5 - Demographic Tension Stress.csv")
INPUT_SHEET = 0
OUTPUT_PREFIX = "DEMOGRAPHIC_TENSION_STRESS_DISTRICT"

# Column names for the new workbook schema
COL_DIST = "ADM2_NAME"
KEEP_ID_COL = "ADM2_NAME"
INDICATOR_COLS = ["Resident_Population (R)"	,"Displaced_Population (D)",	"Heterogeneity (H)"	,"Displacement_Ratio (S = D/R)"	,"Demographic_Factor (DF = S*H)"]


# -----------------------------
# HELPERS
# -----------------------------
def to_numeric_binary_aware(series: pd.Series) -> pd.Series:
    """Coerce numeric values and binary-like booleans to float.

    Handles:
      - Python booleans True/False
      - strings like TRUE/FALSE, true/false, yes/no, y/n, 1/0
      - regular numeric strings
    """
    s = series.copy()

    # Convert to string for tolerant token mapping, preserving NaN.
    s_str = s.astype("string").str.strip().str.lower()
    binary_map = {
        "true": 1,
        "false": 0,
        "yes": 1,
        "no": 0,
        "y": 1,
        "n": 0,
        "1": 1,
        "0": 0,
    }
    mapped = s_str.map(binary_map)

    # Where binary mapping did not apply, fall back to numeric parsing.
    numeric = pd.to_numeric(s, errors="coerce")
    out = mapped.where(mapped.notna(), numeric)

    return out.astype(float)


def minmax01(series: pd.Series) -> pd.Series:
    """Min max normalize to [0, 1]. If constant or invalid, return zeros."""
    s = to_numeric_binary_aware(series)
    smin, smax = np.nanmin(s.values), np.nanmax(s.values)
    if not np.isfinite(smin) or not np.isfinite(smax) or smax == smin:
        return pd.Series(np.zeros(len(s)), index=series.index)
    return (s - smin) / (smax - smin)


def safe_kendall(x: pd.Series, y: pd.Series) -> float:
    """Kendall rank correlation. Returns 0 if undefined or constant."""
    x_ = to_numeric_binary_aware(x)
    y_ = to_numeric_binary_aware(y)

    mask = x_.notna() & y_.notna()
    if mask.sum() < 2:
        return 0.0

    if x_[mask].nunique() < 2 or y_[mask].nunique() < 2:
        return 0.0

    return float(pd.Series(x_[mask]).corr(pd.Series(y_[mask]), method="kendall"))

def load_input_table(path: Path, sheet=0) -> pd.DataFrame:
    """Load CSV or Excel without forcing openpyxl.
    For .xlsx files, tries calamine first, then pandas default engine resolution.
    """
    suffix = path.suffix.lower()

    if suffix == ".csv":
        return pd.read_csv(path)

    if suffix in {".xlsx", ".xlsm", ".xls"}:
        try:
            return pd.read_excel(path, sheet_name=sheet, engine="calamine")
        except Exception:
            try:
                return pd.read_excel(path, sheet_name=sheet)
            except Exception as exc:
                raise RuntimeError(
                    "Could not read Excel file without openpyxl. "
                    "Either install python-calamine (recommended) or save input as CSV."
                ) from exc

    raise ValueError(f"Unsupported input format: {path.suffix}")

# -----------------------------
# LOAD AND VALIDATE
# -----------------------------
df = load_input_table(INPUT_FILE, sheet=INPUT_SHEET)

indicator_cols = [c for c in INDICATOR_COLS if c != KEEP_ID_COL]
has_keep_id = KEEP_ID_COL in df.columns

required_cols = [COL_DIST, *indicator_cols]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# -----------------------------
# PREPARE DATA
# -----------------------------
for c in indicator_cols:
    df[c] = to_numeric_binary_aware(df[c])

df[indicator_cols] = df[indicator_cols].apply(lambda s: s.fillna(s.median()))

# -----------------------------
# NORMALIZATION
# -----------------------------
for c in indicator_cols:
    df[f"norm_{c}"] = minmax01(df[c])

# -----------------------------
# KENDALL CORRELATION MATRIX + WEIGHTS
# -----------------------------
kendall_matrix = pd.DataFrame(index=indicator_cols, columns=indicator_cols, dtype=float)
for c1 in indicator_cols:
    for c2 in indicator_cols:
        if c1 == c2:
            kendall_matrix.loc[c1, c2] = 1.0
        else:
            kendall_matrix.loc[c1, c2] = safe_kendall(df[f"norm_{c1}"], df[f"norm_{c2}"])

# Strength per indicator = mean absolute Kendall tau against all other indicators
mean_abs_corr = {}
for c in indicator_cols:
    other_corrs = [abs(kendall_matrix.loc[c, other]) for other in indicator_cols if other != c]
    mean_abs_corr[c] = float(np.mean(other_corrs)) if other_corrs else 0.0

# Normalize strengths to sum to 1 for final weights
total_strength = sum(mean_abs_corr.values())
if total_strength == 0:
    weights = {c: 1.0 / len(indicator_cols) for c in indicator_cols}
else:
    weights = {c: mean_abs_corr[c] / total_strength for c in indicator_cols}

# Weighted normalized terms and composite
for c in indicator_cols:
    df[f"weight_{c}"] = weights[c]
    df[f"weighted_{c}"] = df[f"norm_{c}"] * weights[c]

df["composite_score"] = df[[f"weighted_{c}" for c in indicator_cols]].sum(axis=1)
df = df.sort_values("composite_score", ascending=False).reset_index(drop=True)
df["rank"] = np.arange(1, len(df) + 1)

# -----------------------------
# EXPORT
# -----------------------------
weights_table = pd.DataFrame({
    "Indicator": indicator_cols,
    "Mean_Abs_Kendall_tau": [mean_abs_corr[c] for c in indicator_cols],
    "Final_Weight": [weights[c] for c in indicator_cols],
})

export_cols = [
    "rank",
    COL_DIST,
    *([KEEP_ID_COL] if has_keep_id else []),
    *indicator_cols,
    *[f"norm_{c}" for c in indicator_cols],
    *[f"weight_{c}" for c in indicator_cols],
    *[f"weighted_{c}" for c in indicator_cols],
    "composite_score",
]

output_dir = INPUT_FILE.parent
scored_csv = output_dir / f"{OUTPUT_PREFIX}_Scored.csv"
weights_csv = output_dir / f"{OUTPUT_PREFIX}_Weights.csv"
kendall_csv = output_dir / f"{OUTPUT_PREFIX}_Kendall_Matrix.csv"

df[export_cols].to_csv(scored_csv, index=False)
weights_table.to_csv(weights_csv, index=False)
kendall_matrix.to_csv(kendall_csv, index=True)

# -----------------------------
# CONSOLE SUMMARY
# -----------------------------
top = df.iloc[0]
print("=== Top District (by composite score) ===")
print(f"Rank: {int(top['rank'])}")
print(f"District: {top[COL_DIST]}")
print(f"Composite score: {top['composite_score']:.4f}")
print("\nWeights (sum=1):")
for c in indicator_cols:
    print(f"  {c}: {weights[c]:.3f}")

print("\nResults saved to:")
print(f"  {scored_csv}")
print(f"  {weights_csv}")
print(f"  {kendall_csv}")


SyntaxError: unmatched ']' (606211709.py, line 16)